In [0]:
import urllib.request
import os
from datetime import datetime, timedelta

# Configuration
VOLUME_PATH = "/Volumes/gharchive_dev/raw/files"
BASE_URL = "https://data.gharchive.org"

# Download today's files (configurable via widgets)
dbutils.widgets.text("date", datetime.utcnow().strftime("%Y-%m-%d"), "Date (YYYY-MM-DD)")
dbutils.widgets.text("hours", "11", "Hours (comma-separated, e.g. 0,1,2 or 11)")

target_date = dbutils.widgets.get("date")
hours = [int(h.strip()) for h in dbutils.widgets.get("hours").split(",")]

print(f"Downloading GH Archive for {target_date}, hours: {hours}")
print(f"Target volume: {VOLUME_PATH}")

for hour in hours:
    filename = f"{target_date}-{hour}.json.gz"
    url = f"{BASE_URL}/{filename}"
    local_path = f"/tmp/{filename}"
    volume_dest = f"{VOLUME_PATH}/{filename}"

    # Check if already downloaded
    try:
        dbutils.fs.ls(volume_dest)
        print(f"SKIP: {filename} already exists in volume")
        continue
    except Exception:
        pass

    print(f"Downloading: {url}")
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req) as response, open(local_path, "wb") as out:
            out.write(response.read())

        # Copy to volume
        dbutils.fs.cp(f"file:{local_path}", volume_dest)
        os.remove(local_path)
        print(f"SUCCESS: {filename} -> {volume_dest}")
    except Exception as e:
        print(f"FAILED: {filename} - {e}")

# List files in volume
print("\nFiles in volume:")
for f in dbutils.fs.ls(VOLUME_PATH):
    print(f"  {f.name} ({f.size:,} bytes)")